# Mobile Money Agent Network — Data Audit, Cleaning & Reporting

## Part 1: Data Loading & Pre-Clean Quality Audit

In [2]:
import os
import numpy as np
import pandas as pd

# Ensure exact seed-44 datasets exist in Colab runtime
if not (os.path.exists("agents.csv") and os.path.exists("agent_transactions.csv")):
    rng = np.random.default_rng(44)
    n = 600
    agents_gen = pd.DataFrame({
        "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
        "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
        "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
    })
    txns_gen = pd.DataFrame({
        "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
        "agent_id": rng.choice(agents_gen["agent_id"], n),
        "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
        "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
        "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d"),
    })
    idx = rng.choice(n, 30, replace=False)
    txns_gen.loc[idx[:15], "amount"] = np.nan
    txns_gen.loc[idx[15:20], "amount"] = "N/A"
    txns_gen.loc[idx[20:25], "amount"] = "250000"
    txns_gen.loc[idx[25:], "txn_date"] = txns_gen.loc[idx[25:], "txn_date"].str.replace("-", "/")
    txns_gen = pd.concat([txns_gen, txns_gen.sample(12, random_state=1)], ignore_index=True)
    txns_gen.loc[rng.choice(len(txns_gen), 8, replace=False), "agent_id"] = "AG099"
    agents_gen.to_csv("agents.csv", index=False)
    txns_gen.to_csv("agent_transactions.csv", index=False)

# Load raw datasets (keep amount as string/object initially so we can audit "N/A" vs NaN)
agents_raw = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv", keep_default_na=False)

print("=== RAW SHAPES ===")
print("agents_raw:", agents_raw.shape)
print("txns_raw:", txns_raw.shape)

# Pre-clean audit counts
exact_dups = txns_raw.duplicated().sum()
dup_txn_ids = txns_raw["txn_id"].duplicated().sum()
blank_amounts = (txns_raw["amount"] == "").sum()
na_string_amounts = (txns_raw["amount"] == "N/A").sum()
numeric_amounts = pd.to_numeric(txns_raw["amount"], errors="coerce")
outlier_amounts = (numeric_amounts >= 100000).sum()
slash_dates = txns_raw["txn_date"].str.contains("/").sum()
unknown_agents = (~txns_raw["agent_id"].isin(agents_raw["agent_id"])).sum()
messy_txn_types = txns_raw["txn_type"].value_counts().to_dict()

print("\n=== PRE-CLEAN DATA QUALITY AUDIT ===")
print(f"1. Exact duplicate rows: {exact_dups} (duplicate txn_ids: {dup_txn_ids})")
print(f"2. Missing amounts: {blank_amounts} empty/NaN + {na_string_amounts} literal 'N/A' = {blank_amounts + na_string_amounts} total")
print(f"3. Extreme outlier amounts (>= 100,000): {outlier_amounts}")
print(f"4. Mixed date formats (slash YYYY/MM/DD instead of YYYY-MM-DD): {slash_dates}")
print(f"5. Orphan/unknown agent_id values (not in agents.csv): {unknown_agents} -> {txns_raw.loc[~txns_raw['agent_id'].isin(agents_raw['agent_id']), 'agent_id'].unique().tolist()}")
print(f"6. Unstandardized txn_type values: {messy_txn_types}")


=== RAW SHAPES ===
agents_raw: (20, 3)
txns_raw: (612, 5)

=== PRE-CLEAN DATA QUALITY AUDIT ===
1. Exact duplicate rows: 11 (duplicate txn_ids: 12)
2. Missing amounts: 15 empty/NaN + 7 literal 'N/A' = 22 total
3. Extreme outlier amounts (>= 100,000): 5
4. Mixed date formats (slash YYYY/MM/DD instead of YYYY-MM-DD): 5
5. Orphan/unknown agent_id values (not in agents.csv): 8 -> ['AG099']
6. Unstandardized txn_type values: {'deposit': 142, 'withdrawal': 131, 'Deposit': 128, 'DEPOSIT ': 111, 'Withdrawal': 100}


### Pre-Clean Audit Summary
* **Raw Dataset Shapes:** `agents.csv` has **20 rows × 3 columns**; `agent_transactions.csv` has **612 rows × 5 columns** (600 base transactions plus 12 appended duplicates).
* **Identified Data Quality Defects (6 categories):**
  1. **Duplicate Transactions:** **12 duplicate `txn_id` records** (11 exact row duplicates plus 1 duplicate whose `agent_id` was subsequently mutated to `AG099`).
  2. **Missing Amounts:** **22 missing values** in `amount` (15 blank/`NaN` cells and 7 literal `"N/A"` strings, including duplicated rows).
  3. **Extreme Outliers:** **5 transactions** with an extreme amount of `250,000` (compared to typical lognormal values around KES 500–10,000).
  4. **Mixed Date Formats:** **5 rows** formatted with slashes (`YYYY/MM/DD`) instead of ISO hyphens (`YYYY-MM-DD`).
  5. **Orphan Agent IDs:** **8 rows** assigned to `"AG099"`, which does not exist in `agents.csv` (`AG001`–`AG020`).
  6. **Unstandardized Transaction Types:** 5 inconsistent casing/whitespace variants (`'deposit'`, `'withdrawal'`, `'Deposit'`, `'DEPOSIT '`, `'Withdrawal'`) instead of two clean categories (`'Deposit'` and `'Withdrawal'`).

### Data Privacy & Governance Assessment (Kenya Data Protection Act, 2019)

Under the **Kenya Data Protection Act (2019)**, operational mobile money transaction logs require strict governance because identifiers such as `agent_id`, transaction references (`txn_id`), and geographic location (`county`) can constitute personal or pseudonymous data when linked to agent registry records. To comply with data minimization and purpose limitation principles, direct customer phone numbers and personal names are excluded from the analytical dataset, while incomplete monetary records are transparently tracked via the `amount_flag` and `amount_imputed` audit columns rather than silently altered. In production, raw and cleaned files (`agent_transactions_clean.csv`) must be protected with **AES-256 encryption at rest** and **TLS 1.2+ encryption in transit**, restricted through **role-based access controls (RBAC)** limited to authorized compliance and analytics personnel, and governed by a **7-year statutory financial retention policy** after which records are securely purged or irreversibly anonymized.

## Part 2: Reusable Cleaning Pipeline (`clean_agent_data`) & Verification

In [4]:
# Import reusable cleaning pipeline from cleaning_pipeline.py
from cleaning_pipeline import clean_agent_data


=== POST-CLEAN VERIFICATION ===
Cleaned shape: (580, 9)
Duplicate txn_ids remaining: 0
Missing amounts remaining: 0
Missing txn_dates remaining: 0
Standardized txn_type counts: {'Deposit': 360, 'Withdrawal': 220}
Valid vs Unknown agent counts:
 is_valid_agent
True     573
False      7
Name: count, dtype: int64
Outlier amount counts:
 is_outlier_amount
False    575
True       5
Name: count, dtype: int64

Data types:
 txn_id                       object
txn_date             datetime64[ns]
agent_id                     object
is_valid_agent                 bool
county                       object
onboarded            datetime64[ns]
txn_type                     object
amount                      float64
is_outlier_amount              bool
dtype: object


,txn_id,txn_date,agent_id,is_valid_agent,county,onboarded,txn_type,amount,is_outlier_amount
0,TX00001,2026-03-03,AG009,True,Nakuru,2024-09-01,Deposit,353.0,False
1,TX00002,2026-04-27,AG020,True,Mombasa,2025-08-01,Deposit,1238.0,False
2,TX00003,2026-04-29,AG008,True,Kisumu,2024-08-01,Deposit,9129.0,False
3,TX00004,2026-04-24,AG009,True,Nakuru,2024-09-01,Deposit,652.0,False
4,TX00005,2026-04-17,AG009,True,Nakuru,2024-09-01,Withdrawal,478.0,False


## Part 3: Executive Summary Report & File Exports

In [5]:
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

# 1. Export cleaned dataset
agent_cleaned.to_csv("outputs/agent_cleaned.csv", index=False)

# 2. Build summary report by county and txn_type (all 580 rows + non-outlier comparison)
summary_all = (
    agent_cleaned.groupby(["county", "txn_type"], as_index=False)
    .agg(
        txn_count=("txn_id", "count"),
        total_amount=("amount", "sum"),
        avg_amount=("amount", "mean"),
        outlier_count=("is_outlier_amount", "sum"),
    )
)

summary_no_outliers = (
    agent_cleaned[~agent_cleaned["is_outlier_amount"]]
    .groupby(["county", "txn_type"], as_index=False)
    .agg(
        clean_txn_count=("txn_id", "count"),
        clean_total_amount=("amount", "sum"),
    )
)

agent_summary = summary_all.merge(summary_no_outliers, on=["county", "txn_type"], how="left")
agent_summary["avg_amount"] = agent_summary["avg_amount"].round(2)
agent_summary = agent_summary.sort_values(["county", "txn_type"]).reset_index(drop=True)
agent_summary.to_csv("outputs/agent_summary.csv", index=False)

# 3. Build formatted multi-sheet Excel workbook with live formulas
wb = Workbook()

# --- Sheet 1: Executive Summary ---
ws_sum = wb.active
ws_sum.title = "Executive Summary"

header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
total_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
total_font = Font(name="Calibri", size=11, bold=True)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

ws_sum["A1"] = "Mobile Money Agent Network — Cleaned Summary Report"
ws_sum["A1"].font = Font(name="Calibri", size=14, bold=True, color="1F4E78")
ws_sum["A2"] = "Generated from seed-44 raw data (580 valid-amount transactions; outliers & AG099 flagged)"
ws_sum["A2"].font = Font(name="Calibri", size=10, italic=True, color="595959")

headers = [
    "County", "Transaction Type", "Txn Count (All)", "Total Amount (KES)",
    "Avg Amount (KES)", "Outlier Count (>=100k)", "Clean Txn Count", "Clean Total (KES)"
]
start_row = 4
for col_idx, h in enumerate(headers, start=1):
    cell = ws_sum.cell(row=start_row, column=col_idx, value=h)
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center" if col_idx > 2 else "left", vertical="center")
    cell.border = thin_border

for r_idx, row in agent_summary.iterrows():
    excel_row = start_row + 1 + r_idx
    values = [
        row["county"], row["txn_type"], int(row["txn_count"]), float(row["total_amount"]),
        float(row["avg_amount"]), int(row["outlier_count"]), int(row["clean_txn_count"]),
        float(row["clean_total_amount"])
    ]
    for c_idx, val in enumerate(values, start=1):
        cell = ws_sum.cell(row=excel_row, column=c_idx, value=val)
        cell.border = thin_border
        if c_idx in (3, 6, 7):
            cell.number_format = "#,##0"
        elif c_idx in (4, 5, 8):
            cell.number_format = "#,##0.00"

total_row = start_row + 1 + len(agent_summary)
first_data = start_row + 1
last_data = total_row - 1

ws_sum.cell(row=total_row, column=1, value="TOTAL").font = total_font
ws_sum.cell(row=total_row, column=2, value="All Types").font = total_font
ws_sum.cell(row=total_row, column=3, value=f"=SUM(C{first_data}:C{last_data})").number_format = "#,##0"
ws_sum.cell(row=total_row, column=4, value=f"=SUM(D{first_data}:D{last_data})").number_format = "#,##0.00"
ws_sum.cell(row=total_row, column=5, value=f"=D{total_row}/C{total_row}").number_format = "#,##0.00"
ws_sum.cell(row=total_row, column=6, value=f"=SUM(F{first_data}:F{last_data})").number_format = "#,##0"
ws_sum.cell(row=total_row, column=7, value=f"=SUM(G{first_data}:G{last_data})").number_format = "#,##0"
ws_sum.cell(row=total_row, column=8, value=f"=SUM(H{first_data}:H{last_data})").number_format = "#,##0.00"

for c_idx in range(1, 9):
    cell = ws_sum.cell(row=total_row, column=c_idx)
    cell.fill = total_fill
    cell.font = total_font
    cell.border = thin_border

for col in ws_sum.columns:
    max_len = max(len(str(cell.value or "")) for cell in col if cell.row >= start_row)
    ws_sum.column_dimensions[get_column_letter(col[0].column)].width = max_len + 5

# --- Sheet 2: Cleaned Transactions ---
ws_clean = wb.create_sheet(title="Cleaned Transactions")
clean_export = agent_cleaned.copy()
clean_export["txn_date"] = clean_export["txn_date"].dt.strftime("%Y-%m-%d")
clean_export["onboarded"] = clean_export["onboarded"].dt.strftime("%Y-%m-%d")

for c_idx, col_name in enumerate(clean_export.columns, start=1):
    cell = ws_clean.cell(row=1, column=c_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for r_idx, row in clean_export.iterrows():
    for c_idx, val in enumerate(row, start=1):
        cell = ws_clean.cell(row=r_idx + 2, column=c_idx, value=None if pd.isna(val) else val)
        if c_idx == 8:
            cell.number_format = "#,##0.00"

for col in ws_clean.columns:
    max_len = max(len(str(cell.value or "")) for cell in col)
    ws_clean.column_dimensions[get_column_letter(col[0].column)].width = max_len + 4

wb.save("excel/agent_network_report.xlsx")

print("Saved: outputs/agent_cleaned.csv, outputs/agent_summary.csv, excel/agent_network_report.xlsx")
agent_summary


Saved: outputs/agent_cleaned.csv, outputs/agent_summary.csv, excel/agent_network_report.xlsx


,county,txn_type,txn_count,total_amount,avg_amount,outlier_count,clean_txn_count,clean_total_amount
0,Kiambu,Deposit,43,334737.0,7784.58,1,42,84737.0
1,Kiambu,Withdrawal,26,75274.0,2895.15,0,26,75274.0
2,Kisumu,Deposit,71,431091.0,6071.70,1,70,181091.0
3,Kisumu,Withdrawal,36,74598.0,2072.17,0,36,74598.0
4,Mombasa,Deposit,68,141880.0,2086.47,0,68,141880.0
5,Mombasa,Withdrawal,61,368592.0,6042.49,1,60,118592.0
6,Nairobi,Deposit,114,536484.0,4706.00,1,113,286484.0
7,Nairobi,Withdrawal,58,154930.0,2671.21,0,58,154930.0
8,Nakuru,Deposit,58,181337.0,3126.50,0,58,181337.0
9,Nakuru,Withdrawal,38,326818.0,8600.47,1,37,76818.0


### Executive Findings & Recommendations
1. **Audit & Reconciliation:** From **612 raw rows**, removing **12 duplicate `txn_id` records** left 600 unique transactions; dropping **20 records with missing amounts** (`15 NaN` and `5 "N/A"`) produced **580 clean, analyzable transactions** (`360 Deposits` and `220 Withdrawals`).
2. **Transaction Volume & Outlier Impact:** Across all 580 valid-amount transactions, gross network volume is **KES 2,654,897** (**KES 1,650,808** in Deposits vs. **KES 1,004,089** in Withdrawals). Five extreme outlier entries of **KES 250,000** each account for **KES 1,250,000** (47.08% of gross volume); excluding those five anomalies yields **KES 1,404,897** across 575 normal transactions.
3. **Regional Performance:** **Nairobi** leads all counties in transaction volume with **172 transactions** totaling **KES 691,414** gross (**KES 441,414** excluding 1 outlier), followed by **Mombasa** (129 transactions; KES 510,472 gross) and **Kisumu** (107 transactions; KES 505,689 gross).
4. **Operational Controls Needed:** Seven valid-amount transactions totaling **KES 27,156** were logged under unregistered agent ID **`AG099`**. Operations should audit `AG099` immediately, enforce foreign-key validation on `agent_id` at POS entry, and add automated threshold alerts for transactions ≥ KES 100,000.

In [10]:
# Run modular cleaning pipeline from cleaning_pipeline.py and inspect cleaning log
from cleaning_pipeline import clean_agent_data

df_clean, cleaning_log = clean_agent_data(txns, agents)
df_clean.to_csv("outputs/agent_transactions_clean.csv", index=False)
df_clean.to_csv("agent_transactions_clean.csv", index=False)

print("=== Cleaning Log Summary ===")
for step, value in cleaning_log.items():
    print(f"{step}: {value}")

df_clean.head()


=== Cleaning Log Summary ===
raw_rows: 612
duplicates_removed: 12
txn_type_standardized: 376
missing_or_invalid_amounts_dropped: 20
mixed_date_formats_normalized: 5
unregistered_agent_rows_flagged: 7
extreme_outliers_flagged: 5
final_clean_rows: 580
amount_flag_breakdown: {'valid': 580, 'missing_or_invalid_dropped': 20}


    txn_id agent_id    txn_type  amount amount_flag  amount_imputed  is_valid_agent  is_outlier_amount
0  TX00001    AG009     Deposit   353.0       valid           False            True              False
1  TX00002    AG020     Deposit  1238.0       valid           False            True              False
2  TX00003    AG008     Deposit  9129.0       valid           False            True              False
3  TX00004    AG009     Deposit   652.0       valid           False            True              False
4  TX00005    AG009  Withdrawal   478.0       valid           False            True              False

### Data Privacy & Governance Considerations

Mobile money transaction logs constitute sensitive financial records governed by data protection regulations such as Kenya's Data Protection Act (2019), requiring strict adherence to data minimization, purpose limitation, and access control throughout the analytics lifecycle. In this pipeline, customer phone numbers and personal identifiers are excluded from analytical exports so transactions are tracked solely via pseudonymous `txn_id` and `agent_id` keys, reducing re-identification risk while preserving full auditability. Cleaned outputs (`outputs/agent_transactions_clean.csv`) and flagged anomaly records—such as unregistered agent `AG099` and high-value transactions (`>= KES 100,000`)—must be stored in encrypted, role-restricted repositories accessible only to authorized compliance and risk-operations personnel, with automated audit trails logging every query or export.